# 26 · Data modeling on Dataverse (Part 2, optional)

## Goal

Model the Dataverse tables the workflows and MCP tools have been querying
since notebook `09` as if they already existed: `crd_supplierspend`,
`crd_supplierperformance`, `crd_supplierrenewal`. Add two security roles
so the app built in `27`-`31` respects the same "the caller sees what
they're allowed to see" boundary `05` taught for SharePoint.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from csx.config import load_settings
from csx.clients import get_application_token
settings = load_settings()
settings.require("DATAVERSE_ENV_URL")
token = get_application_token(settings)
print("Dataverse Web API reachable")


**This notebook is Part 2 — an optional elective track.** Nothing in
`00`-`25` or `T0`-`T9-bonus` depends on it. It's here for instructors who
want to extend the workshop into traditional Power Platform app
development, using the same spine scenario and the same agent.


## Concept

Every workflow and MCP tool from `09` onward has been querying
`crd_supplierspend` / `crd_supplierperformance` as though they were real
tables — for the agent-only curriculum, a mocked response was enough to
teach the workflow/MCP concepts. Building an actual app on top changes
that: an app's gallery and form controls bind to real Dataverse metadata,
so this notebook makes the tables genuinely exist, with the columns the
rest of Part 2 needs.

**Security roles are not optional here.** `05` taught that SharePoint
knowledge sources are trimmed to what the asking user can see; a Dataverse-
backed canvas app needs the equivalent at the table level — a
`Procurement Lead` role with write access to `crd_supplierrenewal`'s
decision fields, and a `Procurement Analyst` role that's read-only on the
same table. `31`'s capstone verification proves this boundary holds for
the app the same way `05` proved it for the agent.


## Build


### The three tables


In [ ]:
from csx.dataverse import get_or_create_table, get_or_create_column

tables = {
    "crd_supplierspend": "Supplier Spend",
    "crd_supplierperformance": "Supplier Performance",
    "crd_supplierrenewal": "Supplier Renewal",
}
for logical_name, display_name in tables.items():
    get_or_create_table(settings, token, logical_name, display_name)


In [ ]:
columns = [
    ("crd_supplierspend", "crd_annualvalue", "Microsoft.Dynamics.CRM.MoneyAttributeMetadata", "Annual Value"),
    ("crd_supplierspend", "crd_deltapct", "Microsoft.Dynamics.CRM.DecimalAttributeMetadata", "Spend Delta %"),
    ("crd_supplierperformance", "crd_latedeliveries", "Microsoft.Dynamics.CRM.IntegerAttributeMetadata", "Late Deliveries"),
    ("crd_supplierperformance", "crd_satisfactionscore", "Microsoft.Dynamics.CRM.IntegerAttributeMetadata", "Satisfaction Score"),
    ("crd_supplierrenewal", "crd_contractenddate", "Microsoft.Dynamics.CRM.DateTimeAttributeMetadata", "Contract End Date"),
    ("crd_supplierrenewal", "crd_decision", "Microsoft.Dynamics.CRM.StringAttributeMetadata", "Decision"),
    ("crd_supplierrenewal", "crd_autorenewalenabled", "Microsoft.Dynamics.CRM.BooleanAttributeMetadata", "Auto-Renewal Enabled"),
]
for table, column, attr_type, display in columns:
    get_or_create_column(settings, token, table, column, attr_type, display)


### Two security roles, not one


In [ ]:
import subprocess
# Illustrative — real role creation typically goes through the admin center
# or a solution-packaged Role.xml; shown here as the two roles' shape so
# the notebook is explicit about what each can do, checkpointed below.
roles = {
    "Procurement Lead": {"crd_supplierrenewal": "ReadWriteAppendAppendTo"},
    "Procurement Analyst": {"crd_supplierrenewal": "ReadOnly"},
}
for role, perms in roles.items():
    print(f"{role}: {perms}")


In [ ]:
from csx.checkpoint import checkpoint
checkpoint(
    name="Procurement Lead and Procurement Analyst security roles created and assigned to test users",
    probe=lambda: input("Both roles created in PPAC/admin center, each assigned to a distinct test user? (y/n): ") == "y",
    remediation="Power Platform admin center > environment > Security roles > New role. Assign Procurement Lead to one test user, Procurement Analyst to another.",
)


### Seed data — the same suppliers the agent already knows about


In [ ]:
from csx.dataverse import upsert_row

upsert_row(settings, token, "crd_supplierspends", {"crd_supplierspend_name": "Meridian Cables"},
           {"crd_annualvalue": 2450000, "crd_deltapct": 0})
upsert_row(settings, token, "crd_supplierspends", {"crd_supplierspend_name": "Northwind Fasteners"},
           {"crd_annualvalue": 610000, "crd_deltapct": 40})
print("seed rows upserted — idempotent, safe to re-run")


## Verify

Same harness, same golden set, every notebook.


In [ ]:
from csx.dataverse import get_row
# Read back one seeded row by its alternate key, to confirm the table is
# genuinely reachable — full read/write round-trip proof comes in 31.
row = get_row(settings, token, "crd_supplierspends", "crd_supplierspend_name='Meridian Cables'")
print(row)


## Cost


In [ ]:
print("Dataverse table/column/row operations don't consume Copilot Credits — those meter agent build/publish/invoke only, tracked separately in Part 1's ledger.")


## Teardown


In [ ]:
print("No teardown — these tables persist for 27-31 and are the same tables the agent's workflows already assumed existed.")
